# Final Model Training: Sentiment Analysis

Model: TF-IDF + Logistic Regression  
Split: 80% train / 20% test  
Dataset: `data/labeled/labeled.parquet` (4,654 rows)

In [ ]:
import pandas as pd
import pickle
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

df = pd.read_parquet('../data/labeled/labeled.parquet')
with open('../models/model_config.json') as f:
    config = json.load(f)
with open('../models/final_model.pkl', 'rb') as f:
    pipeline = pickle.load(f)

print(f'Model: {config["model"]}')
print(f'Train rows: {config["train_size"]}')
print(f'Test rows: {config["test_size"]}')
print(f'Labels: {config["labels"]}')

In [ ]:
# Reproduce train/test split for evaluation
df_labeled = df[df['auto_label'].notna()].copy()
X = df_labeled['text']
y = df_labeled['auto_label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

y_pred = pipeline.predict(X_test)
print(classification_report(y_test, y_pred))

In [ ]:
# Confusion matrix
import os
os.makedirs('../data/model', exist_ok=True)

labels = sorted(y.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)
fig, ax = plt.subplots(figsize=(7, 6))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.savefig('../data/model/confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved confusion_matrix.png')
print(pd.DataFrame(cm, index=labels, columns=labels))

In [ ]:
# Top features (most predictive words)
vectorizer = pipeline.named_steps.get('tfidf') or pipeline.named_steps.get('vectorizer')
clf = pipeline.named_steps.get('clf') or pipeline.named_steps.get('classifier')

if vectorizer and clf and hasattr(clf, 'coef_'):
    feature_names = vectorizer.get_feature_names_out()
    coef = clf.coef_[0]
    top_neg_idx = np.argsort(coef)[:15]
    top_pos_idx = np.argsort(coef)[-15:][::-1]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    axes[0].barh(range(15), coef[top_neg_idx], color='#e74c3c')
    axes[0].set_yticks(range(15))
    axes[0].set_yticklabels(feature_names[top_neg_idx])
    axes[0].invert_yaxis()
    axes[0].set_title('Top 15 features → NEG')
    axes[0].set_xlabel('Coefficient')

    axes[1].barh(range(15), coef[top_pos_idx], color='#2ecc71')
    axes[1].set_yticks(range(15))
    axes[1].set_yticklabels(feature_names[top_pos_idx])
    axes[1].invert_yaxis()
    axes[1].set_title('Top 15 features → POS')
    axes[1].set_xlabel('Coefficient')

    plt.tight_layout()
    plt.savefig('../data/model/top_features.png', dpi=120, bbox_inches='tight')
    plt.close()
    print('Top NEG features:', list(feature_names[top_neg_idx[:5]]))
    print('Top POS features:', list(feature_names[top_pos_idx[:5]]))
else:
    print('Feature visualization not available for this pipeline structure')

In [ ]:
# Final summary
metrics = config['metrics']
print('=== FINAL MODEL METRICS ===')
print(f'Model:      TF-IDF + Logistic Regression')
print(f'Train set:  {config["train_size"]:,} rows')
print(f'Test set:   {config["test_size"]:,} rows')
print(f'Accuracy:   {metrics["accuracy"]:.4f}')
print(f'F1 macro:   {metrics["f1_macro"]:.4f}')
print(f'Precision:  {metrics["precision"]:.4f}')
print(f'Recall:     {metrics["recall"]:.4f}')
print()
print('Model saved: models/final_model.pkl')
print('Usage:')
print('  import pickle')
print('  model = pickle.load(open("models/final_model.pkl", "rb"))')
print('  pred = model.predict(["This movie was absolutely brilliant!"])')
print('  # → ["pos"]')